# CREATE FEATURES

In [353]:
import pandas as pd
import numpy as np

sales = pd.read_csv('../data/processed/sales.csv')
np.random.seed(0)

In [354]:
sales['Date'] = pd.to_datetime(sales['Date'])

In [355]:
print(sales['Date'].dtype)

datetime64[us]


In [356]:
sales = sales.sort_values('Date').reset_index(drop=True)

In [357]:
# Lag features cho Revenue
for lag in [1, 2, 3, 7, 14, 30]:
    sales[f'revenue_lag_{lag}'] = sales['Revenue'].shift(lag)
# Lag features cho COGS
for lag in [1, 7, 30]:
    sales[f'cogs_lag_{lag}'] = sales['COGS'].shift(lag)

In [358]:
sales.head(10)

,Date,Revenue,COGS,is_holiday,log_Revenue,log_COGS,revenue_lag_1,revenue_lag_2,revenue_lag_3,revenue_lag_7,revenue_lag_14,revenue_lag_30,cogs_lag_1,cogs_lag_7,cogs_lag_30
0,2012-07-04,5123547.94,3982991.19,0,15.449358,15.197544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-05,2751773.45,2150580.23,0,14.827757,14.581249,5123547.94,NaN,NaN,NaN,NaN,NaN,3982991.19,NaN,NaN
2,2012-07-06,3054029.42,2517632.84,0,14.931973,14.738830,2751773.45,5123547.94,NaN,NaN,NaN,NaN,2150580.23,NaN,NaN
3,2012-07-07,2667930.94,2108246.62,0,14.796814,14.561368,3054029.42,2751773.45,5123547.94,NaN,NaN,NaN,2517632.84,NaN,NaN
4,2012-07-08,2360851.90,1808622.79,0,14.674534,14.408077,2667930.94,3054029.42,2751773.45,NaN,NaN,NaN,2108246.62,NaN,NaN
5,2012-07-09,3548386.46,2787841.68,0,15.082004,14.840779,2360851.90,2667930.94,3054029.42,NaN,NaN,NaN,1808622.79,NaN,NaN
6,2012-07-10,5234938.62,4044438.84,0,15.470866,15.212854,3548386.46,2360851.90,2667930.94,NaN,NaN,NaN,2787841.68,NaN,NaN
7,2012-07-11,5582884.78,4338313.07,0,15.535216,15.282996,5234938.62,3548386.46,2360851.90,5123547.94,NaN,NaN,4044438.84,3982991.19,NaN
8,2012-07-12,5734632.02,4458811.27,0,15.562034,15.310393,5582884.78,5234938.62,3548386.46,2751773.45,NaN,NaN,4338313.07,2150580.23,NaN
9,2012-07-13,5309511.71,4143402.78,0,15.485011,15.237028,5734632.02,5582884.78,5234938.62,3054029.42,NaN,NaN,4458811.27,2517632.84,NaN


In [359]:
# rolling mean
for w in [7, 14, 30]:
    sales[f'rev_roll{w}_mean'] = sales['Revenue'].shift(1).rolling(w).mean()
    if w in [7, 30]: 
        sales[f'rev_roll{w}_std'] = sales['Revenue'].shift(1).rolling(w).std()
        sales[f'rev_roll{w}_max'] = sales['Revenue'].shift(1).rolling(w).max()

In [360]:
# calendar 
sales['day_of_week'] = sales['Date'].dt.dayofweek 
sales['month'] = sales['Date'].dt.month
sales['day_of_month'] = sales['Date'].dt.day
sales['week_of_year'] = sales['Date'].dt.isocalendar().week.astype(int)
sales['quarter'] = sales['Date'].dt.quarter

sales['is_weekend'] = sales['day_of_week'].isin([5, 6]).astype(int)

fixed_holidays = []

for year in range(2012, 2026):
    fixed_holidays.extend([f'{year}-01-01', f'{year}-04-30', f'{year}-05-01', f'{year}-09-02'])
# gio to hung vuong
gioto_dates = [
    '2012-03-31', '2013-04-19', '2014-04-09', '2015-04-28', '2016-04-16', '2017-04-06', 
    '2018-04-25', '2019-04-14', '2020-04-02', '2021-04-21', '2022-04-10', '2023-04-29', 
    '2024-04-18', '2025-04-07'
]

all_holidays = pd.to_datetime(fixed_holidays + gioto_dates)

sales['is_holiday'] = sales['Date'].apply(lambda x: 1 if min(abs((x - h).days) for h in all_holidays) <= 3 else 0)

# Tet
tet_dates = pd.to_datetime([
    '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08', '2017-01-28', 
    '2018-02-16', '2019-02-05', '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-22', 
    '2024-02-10', '2025-01-29'
])

sales['days_to_tet'] = sales['Date'].apply(lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=30)

sales['pre_tet_7d']  = sales['Date'].apply(lambda x: 1 if any(0 < (t - x).days <= 7  for t in tet_dates) else 0)
sales['pre_tet_14d'] = sales['Date'].apply(lambda x: 1 if any(7 < (t - x).days <= 14 for t in tet_dates) else 0)
sales['on_tet']      = sales['Date'].apply(lambda x: 1 if any(abs((x - t).days) <= 1  for t in tet_dates) else 0)
sales['post_tet']    = sales['Date'].apply(lambda x: 1 if any(0 < (x - t).days <= 7  for t in tet_dates) else 0)

In [361]:
# promotion features - khuyen mai
promotions = pd.read_csv('../data/processed/promotions.csv')
promotions['start_date'] = pd.to_datetime(promotions['start_date'], format='%Y-%m-%d')
promotions['end_date'] = pd.to_datetime(promotions['end_date'], format='%Y-%m-%d')

sales['promo_count'] = 0
for _, row in promotions.iterrows():
    mask = (sales['Date'] >= row['start_date']) & (sales['Date'] <= row['end_date'])
    sales.loc[mask, 'promo_count'] += 1

sales['promo_active'] = (sales['promo_count'] > 0).astype(int) # co promo hay khong

sales['promo_intensity'] = sales['promo_count'].clip(upper=5) # max promo la 5
sales['post_promo'] = ((sales['promo_active'].shift(1).fillna(0) == 1) & 
                       (sales['promo_active'] == 0)).astype(int) # ngay sau promo


In [362]:
# web_traffic (du bao doanh so)
# thay co video quang cao -> vao xem -> hoi ban be -> hom sau mua
web_traffic = pd.read_csv('../data/processed/web_traffic.csv')
web_traffic['date'] = pd.to_datetime(web_traffic['date'])

daily_web = web_traffic.groupby('date')['sessions'].sum().reset_index()
sales = sales.merge(daily_web, left_on='Date', right_on='date', how='left').drop(columns=['date'])

sales['sessions_lag7'] = sales['sessions'].shift(7)
sales['sessions_lag14'] = sales['sessions'].shift(14)
sales['sessions_roll7_mean'] = sales['sessions'].shift(1).rolling(7).mean()

roll30 = sales['sessions'].shift(1).rolling(30).mean()
sales['sessions_vs_avg'] = (sales['sessions'] / roll30).fillna(1.0)
sales['sessions_spike'] = (sales['sessions_vs_avg'] > 1.5).astype(int)

In [363]:
print(sales.shape)

print("NaN head(30):")
missing_check = sales[['revenue_lag_30', 'cogs_lag_30', 'sessions_lag14']].isna().sum()
print(missing_check)

sales.to_csv('../data/processed/features_train.csv', index=False)

(3833, 43)
NaN head(30):
revenue_lag_30     30
cogs_lag_30        30
sessions_lag14    195
dtype: int64


# XGBoost + LightGBM + Prophet

## ver1

In [364]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

features_train = pd.read_csv('../data/processed/features_train.csv')
features_train['Date'] = pd.to_datetime(features_train['Date'])

features_train = features_train.dropna().reset_index(drop=True)

In [365]:
# Features X
leakage_cols = ['Date', 'Revenue', 'COGS', 'log_Revenue', 'log_COGS', 'sessions', 'promo_count', 'promo_active']
features = [col for col in features_train.columns if col not in leakage_cols]
X = features_train[features]
y_rev = features_train['Revenue']
y_cogs = features_train['COGS']

# TimeSeriesSplit(n_splits=5)
tscv = TimeSeriesSplit(n_splits=5)

for train_index, val_index in tscv.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train_rev, y_val_rev = y_rev.iloc[train_index], y_rev.iloc[val_index]
    y_train_cogs, y_val_cogs = y_cogs.iloc[train_index], y_cogs.iloc[val_index]

print(f"Train: {X_train.shape}\nVal: {X_val.shape}")

Train: (3032, 35)
Val: (606, 35)


In [366]:
# Train XGBoost
xgb_rev = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42, 
    early_stopping_rounds=50
)
xgb_rev.fit(
    X_train, y_train_rev, 
    eval_set=[(X_val, y_val_rev)], 
    verbose=100
)

xgb_cogs = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42,
    early_stopping_rounds=50
)
xgb_cogs.fit(
    X_train, y_train_cogs, 
    eval_set=[(X_val, y_val_cogs)], 
    verbose=100
)

[0]	validation_0-rmse:2126041.61779
[100]	validation_0-rmse:794027.14796
[200]	validation_0-rmse:780045.49184
[253]	validation_0-rmse:780159.17573
[0]	validation_0-rmse:1764848.47824
[100]	validation_0-rmse:686368.94420
[200]	validation_0-rmse:676291.39871
[262]	validation_0-rmse:680305.05950


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [367]:
# MAE, RMSE, R^2 tai X_val
def calc_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}(MAE, RMSE, R²)] {mae:.2f},{rmse:.2f},{r2:.4f}")

pred_val_rev_xgb = xgb_rev.predict(X_val)
pred_val_cogs_xgb = xgb_cogs.predict(X_val)
calc_metrics(y_val_rev, pred_val_rev_xgb, "XGBoost Revenue")
calc_metrics(y_val_cogs, pred_val_cogs_xgb, "XGBoost COGS")

[XGBoost Revenue(MAE, RMSE, R²)] 547423.53,779749.46,0.7645
[XGBoost COGS(MAE, RMSE, R²)] 470822.91,675197.38,0.7635


In [368]:
# features (test period)
sample_sub = pd.read_csv('../data/processed/sample_submission.csv')
sample_sub['Date'] = pd.to_datetime(sample_sub['Date'])
last_30_train = features_train.tail(30).copy()
test_ft_mock = sample_sub.copy()
concat_ft = pd.concat([last_30_train, test_ft_mock], ignore_index=True)

for col in features:
    if col not in concat_ft.columns:
        concat_ft[col] = np.nan 
X_test = concat_ft[concat_ft['Date'] >= '2023-01-01'][features]
print(f"Test features shape: {X_test.shape}")

Test features shape: (548, 35)


In [369]:
# Predict Revenue, COGS (548 ngày test)
test_preds_rev = xgb_rev.predict(X_test)
test_preds_cogs = xgb_cogs.predict(X_test)

test_preds_rev = np.round(np.clip(test_preds_rev, 0, None), 2)
test_preds_cogs = np.round(np.clip(test_preds_cogs, 0, None), 2)

submission = sample_sub.copy()
submission['Revenue'] = test_preds_rev
submission['COGS'] = test_preds_cogs

print(f"check line number (548): {len(submission)}")
assert len(submission) == 548, "ERROR"

check line number (548): 548


In [370]:
submission.to_csv('../submissions/submission_v1.csv', index=False)
print("Saved successfully: submissions/submission_v1.csv")

Saved successfully: submissions/submission_v1.csv


## ver2

In [371]:
import lightgbm as lgb
from prophet import Prophet

lgb_rev = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.03,
    num_leaves=31,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb_rev.fit(
    X_train, y_train_rev, 
    eval_set=[(X_val, y_val_rev)], 
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

lgb_cogs = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.03,
    num_leaves=31,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb_cogs.fit(
    X_train, y_train_cogs, 
    eval_set=[(X_val, y_val_cogs)], 
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000419 seconds.
You can set `force_col_wise=true` to remove the overhead.


[LightGBM] [Info] Total Bins 5146
[LightGBM] [Info] Number of data points in the train set: 3032, number of used features: 34
[LightGBM] [Info] Start training from score 4556806.431677
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[759]	valid_0's l2: 5.58808e+11
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5146
[LightGBM] [Info] Number of data points in the train set: 3032, number of used features: 34
[LightGBM] [Info] Start training from score 3925446.614910
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[566]	valid_0's l2: 4.49663e+11


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,30


In [372]:
pred_val_rev_lgb = lgb_rev.predict(X_val)
pred_val_cogs_lgb = lgb_cogs.predict(X_val)

calc_metrics(y_val_rev, pred_val_rev_lgb, "LightGBM Revenue")
calc_metrics(y_val_cogs, pred_val_cogs_lgb, "LightGBM COGS")

[LightGBM Revenue(MAE, RMSE, R²)] 531933.41,747534.80,0.7836
[LightGBM COGS(MAE, RMSE, R²)] 465381.45,670568.83,0.7668


In [373]:
ens_val_rev = (pred_val_rev_xgb + pred_val_rev_lgb) / 2
ens_val_cogs = (pred_val_cogs_xgb + pred_val_cogs_lgb) / 2

calc_metrics(y_val_rev, ens_val_rev, "Ensemble (XGB + LGB) Revenue")
calc_metrics(y_val_cogs, ens_val_cogs, "Ensemble (XGB + LGB) COGS")

[Ensemble (XGB + LGB) Revenue(MAE, RMSE, R²)] 533570.56,754957.28,0.7793
[Ensemble (XGB + LGB) COGS(MAE, RMSE, R²)] 463390.88,666847.18,0.7694


In [374]:
ens_val_rev = (pred_val_rev_xgb * 0.4 + pred_val_rev_lgb * 0.6)
ens_val_cogs = (pred_val_cogs_xgb * 0.7 + pred_val_cogs_lgb * 0.3)

calc_metrics(y_val_rev, ens_val_rev, "Ensemble (4XGB + 6LGB) Revenue")
calc_metrics(y_val_cogs, ens_val_cogs, "Ensemble (7XGB + 3LGB) COGS")

[Ensemble (4XGB + 6LGB) Revenue(MAE, RMSE, R²)] 532138.46,752049.42,0.7810
[Ensemble (7XGB + 3LGB) COGS(MAE, RMSE, R²)] 465302.50,668749.32,0.7680


In [375]:
from sklearn.model_selection import ParameterGrid

param_grid = {
    'w_a': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
}

best_score = float('inf')
best_weights = None
def calc_mae(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    return mae

def calc_rmse(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse

for p in ParameterGrid(param_grid):
    w_a = p['w_a']
    w_b = 1.0 - w_a 
    
    preds_rev = (w_a * pred_val_rev_xgb) + (w_b * pred_val_rev_lgb)
    preds_cogs = (w_a * pred_val_cogs_xgb) + (w_b * pred_val_cogs_lgb)
    score_rev = calc_rmse(y_val_rev, preds_rev, "Ensemble Revenue")
    score_cogs = calc_mae(y_val_cogs, preds_cogs, "Ensemble COGS")

    if score_rev < best_score:
        best_score = score_rev
        best_weights_rev = (w_a, w_b)
    if score_cogs < best_score:
        best_score = score_cogs
        best_weights_cogs = (w_a, w_b)

print(f"Best Weights (Revenue): {best_weights_rev}")
print(f"Best Weights (COGS): {best_weights_cogs}")

Best Weights (Revenue): (0.1, 0.9)
Best Weights (COGS): (0.4, 0.6)


In [376]:
ens_val_rev = (pred_val_rev_xgb * 0.1 + pred_val_rev_lgb * 0.9)
ens_val_cogs = (pred_val_cogs_xgb * 0.2 + pred_val_cogs_lgb * 0.8)

calc_metrics(y_val_rev, ens_val_rev, "Ensemble (1XGB + 9LGB) Revenue")
calc_metrics(y_val_cogs, ens_val_cogs, "Ensemble (2XGB + 8LGB) COGS")

[Ensemble (1XGB + 9LGB) Revenue(MAE, RMSE, R²)] 531513.65,747587.45,0.7836
[Ensemble (2XGB + 8LGB) COGS(MAE, RMSE, R²)] 463834.76,667629.80,0.7688


In [377]:
print("Prophet:")

train_rev_df = features_train.loc[X_train.index, ['Date', 'Revenue']].rename(columns={'Date': 'ds', 'Revenue': 'y'})
m_rev = Prophet()
#m_rev.add_country_holidays(country_name='VN')   
m_rev.fit(train_rev_df)

train_cogs_df = features_train.loc[X_train.index, ['Date', 'COGS']].rename(columns={'Date': 'ds', 'COGS': 'y'})
m_cogs = Prophet()
#m_cogs.add_country_holidays(country_name='VN')
m_cogs.fit(train_cogs_df)

val_dates = features_train.loc[X_val.index, ['Date']].rename(columns={'Date': 'ds'})

prophet_preds_rev = m_rev.predict(val_dates)
prophet_preds_cogs = m_cogs.predict(val_dates)

pred_val_rev_prophet = prophet_preds_rev['yhat'].values
pred_val_cogs_prophet = prophet_preds_cogs['yhat'].values

calc_metrics(y_val_rev, pred_val_rev_prophet, "Prophet Revenue")
calc_metrics(y_val_cogs, pred_val_cogs_prophet, "Prophet COGS")

Prophet:


17:31:19 - cmdstanpy - INFO - Chain [1] start processing
17:31:19 - cmdstanpy - INFO - Chain [1] done processing
17:31:19 - cmdstanpy - INFO - Chain [1] start processing
17:31:19 - cmdstanpy - INFO - Chain [1] done processing


[Prophet Revenue(MAE, RMSE, R²)] 1019405.58,1357281.17,0.2866
[Prophet COGS(MAE, RMSE, R²)] 967278.19,1268403.79,0.1655


In [378]:
import optuna
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def objective_rev(trial):
    w_a = trial.suggest_float('w_a', 0.0, 1.0)
    w_b = trial.suggest_float('w_b', 0.0, 1.0)
    w_c = trial.suggest_float('w_c', 0.0, 1.0)
    total = w_a + w_b + w_c
    w_a, w_b, w_c = w_a/total, w_b/total, w_c/total
    preds = (w_a * pred_val_rev_xgb) + (w_b * pred_val_rev_lgb) + (w_c * pred_val_rev_prophet)
    '''mse = mean_squared_error(y_val_rev, preds)
    return np.sqrt(mse)'''
    return mean_absolute_error(y_val_rev, preds)

def objective_cogs(trial):
    w_a = trial.suggest_float('w_a', 0.0, 1.0)
    w_b = trial.suggest_float('w_b', 0.0, 1.0)
    w_c = trial.suggest_float('w_c', 0.0, 1.0)
    total = w_a + w_b + w_c
    w_a, w_b, w_c = w_a/total, w_b/total, w_c/total
    preds = (w_a * pred_val_cogs_xgb) + (w_b * pred_val_cogs_lgb) + (w_c * pred_val_cogs_prophet)
    '''return mean_absolute_error(y_val_cogs, preds)'''
    mse = mean_squared_error(y_val_cogs, preds)
    return np.sqrt(mse)

study_rev = optuna.create_study(direction='minimize')
study_rev.optimize(objective_rev, n_trials=100)

study_cogs = optuna.create_study(direction='minimize')
study_cogs.optimize(objective_cogs, n_trials=100)

def get_weights(study):
    params = study.best_params
    w = np.array([params['w_a'], params['w_b'], params['w_c']])
    return w / w.sum()

weights_rev = get_weights(study_rev)
weights_cogs = get_weights(study_cogs)

print(f"Revenue (XGB, LGB, Prophet): {weights_rev}")
print(f"COGS (XGB, LGB, Prophet): {weights_cogs}")

[I 2026-04-24 17:31:20,035] A new study created in memory with name: no-name-b3ec69dd-ffad-4f84-a6a1-a82d317ea6df
[I 2026-04-24 17:31:20,038] Trial 0 finished with value: 623248.2187932964 and parameters: {'w_a': 0.3944988118509076, 'w_b': 0.7771913056577251, 'w_c': 0.9095068576567225}. Best is trial 0 with value: 623248.2187932964.
[I 2026-04-24 17:31:20,039] Trial 1 finished with value: 554566.744941644 and parameters: {'w_a': 0.852556751276211, 'w_b': 0.5884144656087932, 'w_c': 0.5106374819192196}. Best is trial 1 with value: 554566.744941644.
[I 2026-04-24 17:31:20,040] Trial 2 finished with value: 593591.9270380284 and parameters: {'w_a': 0.7891590586140002, 'w_b': 0.805270161683433, 'w_c': 0.9405967575949022}. Best is trial 1 with value: 554566.744941644.
[I 2026-04-24 17:31:20,042] Trial 3 finished with value: 664445.5583560165 and parameters: {'w_a': 0.3065357609406255, 'w_b': 0.5665395602707418, 'w_c': 0.910065833984178}. Best is trial 1 with value: 554566.744941644.
[I 2026-0

Revenue (XGB, LGB, Prophet): [0.06036197 0.8309693  0.10866874]
COGS (XGB, LGB, Prophet): [0.37941676 0.5510083  0.06957494]


In [379]:
ens_val_rev = (pred_val_rev_xgb * 0.35 + pred_val_rev_lgb * 0.55 + pred_val_rev_prophet * 0.10)
ens_val_cogs = (pred_val_cogs_xgb * 0.65 + pred_val_cogs_lgb * 0.25 + pred_val_cogs_prophet * 0.10)

calc_metrics(y_val_rev, ens_val_rev, "Ensemble Revenue 3 Models ")
calc_metrics(y_val_cogs, ens_val_cogs, "Ensemble COGS 3 Models ")

[Ensemble Revenue 3 Models (MAE, RMSE, R²)] 523256.11,741746.48,0.7869
[Ensemble COGS 3 Models (MAE, RMSE, R²)] 462385.30,664996.33,0.7706


In [380]:
# MAE - RMSE
ens_val_rev = (pred_val_rev_xgb * 0.04237056 + pred_val_rev_lgb * 0.84849617 + pred_val_rev_prophet * 0.10913327)
ens_val_cogs = (pred_val_cogs_xgb * 0.34022152 + pred_val_cogs_lgb * 0.59144053 + pred_val_cogs_prophet * 0.06833796)

calc_metrics(y_val_rev, ens_val_rev, "Ensemble Revenue 3 Models ")
calc_metrics(y_val_cogs, ens_val_cogs, "Ensemble COGS 3 Models ")

[Ensemble Revenue 3 Models (MAE, RMSE, R²)] 520058.97,737588.70,0.7893
[Ensemble COGS 3 Models (MAE, RMSE, R²)] 458320.47,661915.49,0.7727


In [381]:
# RMSE - MAE
ens_val_rev = (pred_val_rev_xgb * 0.22683048 + pred_val_rev_lgb * 0.6777754 + pred_val_rev_prophet * 0.09539412)
ens_val_cogs = (pred_val_cogs_xgb * 0.12274082 + pred_val_cogs_lgb * 0.81408858 + pred_val_cogs_prophet * 0.06317059)

calc_metrics(y_val_rev, ens_val_rev, "Ensemble Revenue 3 Models ")
calc_metrics(y_val_cogs, ens_val_cogs, "Ensemble COGS 3 Models ")

[Ensemble Revenue 3 Models (MAE, RMSE, R²)] 521488.57,739163.21,0.7884
[Ensemble COGS 3 Models (MAE, RMSE, R²)] 458587.41,663467.25,0.7717


In [382]:
pred_test_rev_xgb = xgb_rev.predict(X_test)
pred_test_rev_lgb = lgb_rev.predict(X_test)
pred_test_cogs_xgb = xgb_cogs.predict(X_test)
pred_test_cogs_lgb = lgb_cogs.predict(X_test)

test_dates = sample_sub[['Date']].rename(columns={'Date': 'ds'})
prophet_test_preds_rev = m_rev.predict(test_dates)
prophet_test_preds_cogs = m_cogs.predict(test_dates)
pred_test_rev_prophet = prophet_test_preds_rev['yhat'].values
pred_test_cogs_prophet = prophet_test_preds_cogs['yhat'].values

final_test_rev = (pred_test_rev_xgb * 0.35 + pred_test_rev_lgb * 0.55 + pred_test_rev_prophet * 0.10)
final_test_cogs = (pred_test_cogs_xgb * 0.65 + pred_test_cogs_lgb * 0.25 + pred_test_cogs_prophet * 0.10)

final_test_rev = np.round(np.clip(final_test_rev, 0, None), 2)
final_test_cogs = np.round(np.clip(final_test_cogs, 0, None), 2)

submission_v2 = sample_sub.copy()
submission_v2['Revenue'] = final_test_rev
submission_v2['COGS'] = final_test_cogs

print(f"check line number (548): {len(submission_v2)}")
assert len(submission_v2) == 548, "ERROR"

check line number (548): 548


In [383]:
submission_v2.to_csv('../submissions/submission_v2.csv', index=False)
print("Saved: ../submissions/submission_v2.csv")

Saved: ../submissions/submission_v2.csv


# SHAP ANALYSIS

In [384]:
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

best_model = lgb_rev
explainer = shap.TreeExplainer(best_model) 

In [385]:
shap_values = explainer.shap_values(X_val)

print(f"X_val shape: {X_val.shape}")
print(f"shap_values shape: {shap_values.shape}")

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_val.columns
).sort_values(ascending=False)

print("\nTop 10 features (SHAP):")
print(mean_abs_shap.head(10))

X_val shape: (606, 35)
shap_values shape: (606, 35)

Top 10 features (SHAP):
revenue_lag_1     1.144051e+06
day_of_month      3.278279e+05
revenue_lag_7     2.283618e+05
revenue_lag_14    1.307801e+05
cogs_lag_1        1.123783e+05
cogs_lag_7        1.023170e+05
day_of_week       8.331963e+04
week_of_year      7.680453e+04
revenue_lag_2     7.346642e+04
cogs_lag_30       5.389754e+04
dtype: float64


In [386]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_val,
    plot_type="dot",
    max_display=20,
    show=False
)
plt.title("SHAP Summary Plot — Revenue Model (LightGBM)", fontsize=13)
plt.tight_layout()
plt.savefig('../report/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: ../report/figures/shap_summary.png")

Saved: ../report/figures/shap_summary.png


In [387]:
plt.figure(figsize=(10, 8))

shap.plots.bar(
    shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value,
        data=X_val.values,
        feature_names=X_val.columns.tolist()
    ),
    max_display=20,
    show=False
)

plt.title("SHAP Feature Importance — Revenue Model", fontsize=13)
plt.tight_layout()
plt.savefig('../report/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: ../report/figures/feature_importance.png")

Saved: ../report/figures/feature_importance.png


In [388]:
print("TOP 15 FEATURES:")
tet_feature_candidates = {'is_tet', 'pre_tet_7d', 'pre_tet_14d', 'on_tet', 'post_tet', 'days_to_tet'}
other_expected = {'revenue_lag_7', 'sessions_lag7', 'promo_intensity'}

for i, (feat, val) in enumerate(mean_abs_shap.head(15).items(), 1):
    flag = "(*)" if feat in (tet_feature_candidates | other_expected) else ""
    print(f"{i:2d}. {feat:<30} {val:>10.2f}{flag}")

# check expected features in top 10
actual_top10 = set(mean_abs_shap.head(10).index)
found_tet_features = tet_feature_candidates & actual_top10
missing_other_expected = other_expected - actual_top10

if found_tet_features:
    print(f"\nFeature Tết vào top 10: {sorted(found_tet_features)}")
else:
    print("\nChưa có feature Tết nào vào top 10")

if missing_other_expected:
    print(f"Features kỳ vọng khác chưa vào top 10: {sorted(missing_other_expected)}")
else:
    print("Tất cả feature kỳ vọng khác đều vào top 10")

TOP 15 FEATURES:
 1. revenue_lag_1                  1144051.11
 2. day_of_month                    327827.89
 3. revenue_lag_7                   228361.84(*)
 4. revenue_lag_14                  130780.06
 5. cogs_lag_1                      112378.33
 6. cogs_lag_7                      102316.99
 7. day_of_week                      83319.63
 8. week_of_year                     76804.53
 9. revenue_lag_2                    73466.42
10. cogs_lag_30                      53897.54
11. rev_roll30_mean                  51755.05
12. promo_intensity                  50331.23(*)
13. rev_roll7_std                    44677.11
14. sessions_vs_avg                  41665.22
15. rev_roll7_mean                   32511.09

Chưa có feature Tết nào vào top 10
Features kỳ vọng khác chưa vào top 10: ['promo_intensity', 'sessions_lag7']
